In [1]:
import cv2
import os
import numpy as np
import zipfile

zip_path = r"C:\Users\CCB\Data_set.zip"
extract_path = r"C:\Users\CCB\Data_set_extracted"

if not os.path.exists(zip_path):
    raise FileNotFoundError(f"ZIP file not found: {zip_path}")

os.makedirs(extract_path, exist_ok=True)

if not os.listdir(extract_path):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

print("Dataset folder:", extract_path)
print("OpenCV version:", cv2.__version__)
print("LBPH available:", hasattr(cv2.face, "LBPHFaceRecognizer_create"))

Dataset folder: C:\Users\CCB\Data_set_extracted
OpenCV version: 4.10.0
LBPH available: True


In [2]:
from collections import defaultdict

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

faces = []
labels = []
label_names = {}

image_extensions = (".jpg", ".jpeg", ".png", ".bmp")
label_id = 0

for root, dirs, files in os.walk(extract_path):
    # Skip folders without images
    image_files = [
        f for f in files
        if f.lower().endswith(image_extensions)
    ]

    if not image_files:
        continue

    person_name = os.path.basename(root)

    # Ignore folders containing only saved model files
    person_faces = []

    for filename in image_files:
        image_path = os.path.join(root, filename)
        image = cv2.imread(image_path)

        if image is None:
            continue

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        detected_faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(40, 40)
        )

        if len(detected_faces) == 0:
            continue

        # Select the largest face
        x, y, w, h = max(
            detected_faces,
            key=lambda box: box[2] * box[3]
        )

        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (200, 200))

        person_faces.append(face)

    if person_faces:
        label_names[label_id] = person_name
        faces.extend(person_faces)
        labels.extend([label_id] * len(person_faces))

        print(f"{person_name}: {len(person_faces)} faces")
        label_id += 1

if len(faces) == 0:
    raise ValueError(
        "No faces were detected. Check your dataset folder structure."
    )

faces = np.array(faces, dtype=np.uint8)
labels = np.array(labels, dtype=np.int32)

print("\nDataset preparation completed!")
print("Total face images:", len(faces))
print("Total people:", len(label_names))
print("Face array shape:", faces.shape)
print("Label array shape:", labels.shape)

person1: 9 faces
person2: 9 faces

Dataset preparation completed!
Total face images: 18
Total people: 2
Face array shape: (18, 200, 200)
Label array shape: (18,)


In [3]:
import json

if not hasattr(cv2, "face"):
    raise ImportError(
        "OpenCV face module is missing. Check your OpenCV installation."
    )

if len(faces) == 0 or len(labels) == 0:
    raise ValueError(
        "No training data found. Run Step 2 first."
    )

recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=1,
    neighbors=8,
    grid_x=8,
    grid_y=8
)

recognizer.train(faces, labels)

model_path = os.path.join(extract_path, "face_model.yml")
recognizer.save(model_path)

names_path = os.path.join(extract_path, "label_names.json")

with open(names_path, "w", encoding="utf-8") as file:
    json.dump(
        {str(k): v for k, v in label_names.items()},
        file,
        ensure_ascii=False,
        indent=4
    )

print("Face recognition model trained successfully!")
print("Model saved at:", model_path)
print("Person names saved at:", names_path)

Face recognition model trained successfully!
Model saved at: C:\Users\CCB\Data_set_extracted\face_model.yml
Person names saved at: C:\Users\CCB\Data_set_extracted\label_names.json


In [4]:
print("Model exists:", os.path.exists(model_path))
print("Number of training images:", len(faces))
print("Recognized identity classes:", label_names)

Model exists: True
Number of training images: 18
Recognized identity classes: {0: 'person1', 1: 'person2'}


In [ ]:

import cv2
import os
import json

# Paths to the trained model and saved names
model_path = os.path.join(extract_path, "face_model.yml")
names_path = os.path.join(extract_path, "label_names.json")

# Check that the trained files exist
if not os.path.exists(model_path):
    raise FileNotFoundError("Trained model not found. Run the training cell first.")

if not os.path.exists(names_path):
    raise FileNotFoundError("Name mapping not found. Run the training cell first.")

# Load the trained recognizer
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read(model_path)

# Load person names
with open(names_path, "r", encoding="utf-8") as file:
    label_names = {
        int(k): v for k, v in json.load(file).items()
    }

# Load face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Open the laptop camera
camera = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not camera.isOpened():
    camera.release()
    camera = cv2.VideoCapture(0)

if not camera.isOpened():
    raise RuntimeError(
        "Camera not found. Close other camera apps or try camera index 1."
    )

print("Camera started! Press Q on the camera window to exit.")

try:
    while True:
        ret, frame = camera.read()

        if not ret:
            print("Unable to read from camera.")
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        detected_faces = face_detector.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(60, 60)
        )

        for (x, y, w, h) in detected_faces:
            face = gray[y:y+h, x:x+w]
            face = cv2.resize(face, (200, 200))

            predicted_label, confidence = recognizer.predict(face)

            # Lower LBPH confidence means a closer match
            if predicted_label in label_names and confidence < 65:
                name = label_names[predicted_label]
                color = (0, 255, 0)  # Green: recognized
            else:
                name = "Unknown"
                color = (0, 0, 255)  # Red: unknown

            cv2.rectangle(
                frame, (x, y), (x+w, y+h), color, 2
            )

            cv2.putText(
                frame,
                f"{name} ({confidence:.1f})",
                (x, max(y - 10, 25)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2
            )

        cv2.imshow("Real-Time Face Recognition", frame)

        # Press Q while the camera window is active
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

finally:
    camera.release()
    cv2.destroyAllWindows()


Camera started! Press Q on the camera window to exit.
